# RFESelector

Recursive feature elimination with a linear SVM (SVM-RFE, Guyon et al. 2002).
Fit the SVM, drop the features with the smallest absolute weights, refit on
the rest, and repeat until `n_features` remain.

- **Cost:** one SVM fit per round. `step=0.1` removes a tenth of the starting
  features per round, so about ten rounds.
- **Strength:** each feature is judged alongside the others, so a feature that
  only helps in combination can survive.
- **Weakness:** a linear model's weights only see linear signal, and the result
  depends on the SVM's regularization `C`.
- **Two classes only.** Selects the same features as scikit-learn's
  `RFE(SVC(kernel='linear'))`, faster when samples are far fewer than
  features: the SVM is trained on the samples' Gram matrix.

In [1]:
import time

import numpy as np
from sklearn.datasets import make_classification

rng = np.random.default_rng(0)

# 500 features, the first 10 informative (shuffle=False keeps them first).
X, y = make_classification(n_samples=300, n_features=500, n_informative=10,
                           n_redundant=0, shuffle=False, random_state=0)
truth = set(range(10))

from featsel.selectors import RFESelector

## Regularization first

With more features than samples, the classes can be separated perfectly, and
a weakly regularized SVM spreads its weights over noise. Its ranking is then
close to arbitrary. A small `C` (strong regularization) fixes that. For
reference, ANOVAFSelector finds 5 of the 10 on this data.

In [2]:
for C in (1.0, 0.1, 0.01, 0.001):
    chosen = RFESelector(n_features=10, C=C).fit(X, y).get_support(indices=True)
    print(f'C={C:<6} truly informative: {len(truth & set(chosen))} of 10')

C=1.0    truly informative: 3 of 10
C=0.1    truly informative: 3 of 10


C=0.01   truly informative: 5 of 10
C=0.001  truly informative: 5 of 10


## `step` sets the time

A larger step removes more features per round, so fewer SVM fits. With a
well-regularized SVM it changes the time far more than the result.

In [3]:
for step in (0.5, 0.1, 0.02):
    start = time.perf_counter()
    chosen = RFESelector(n_features=10, step=step, C=0.01).fit(X, y).get_support(indices=True)
    print(f'step={step:<5} {time.perf_counter() - start:5.2f} s   '
          f'truly informative: {len(truth & set(chosen))} of 10')

step=0.5    0.03 s   truly informative: 5 of 10
step=0.1    0.10 s   truly informative: 5 of 10


step=0.02   0.27 s   truly informative: 5 of 10


## Where it fails

Here the class depends on how *far* a feature is from zero, not on its sign.
Both classes have the same mean, so no linear weight separates them and the feature ranks like noise. One draw can land
anywhere, so the cell averages 20.

In [4]:
ranks = []
for seed in range(20):
    draw = np.random.default_rng(seed)
    x = draw.normal(size=1000)
    y_far = (np.abs(x) > 0.6745).astype(int)  # the median of |x|: balanced classes
    X_far = np.column_stack([x, draw.normal(size=(1000, 49))])
    ranks.append(RFESelector(n_features=5, step=1).fit(X_far, y_far).ranking_[0])
print('mean elimination rank of the one informative feature over 20 draws:',
      np.mean(ranks), '(1 = kept, 46 = removed first; noise averages about 22)')

mean elimination rank of the one informative feature over 20 draws: 21.9 (1 = kept, 46 = removed first; noise averages about 22)
